# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [48]:
# Write your code below.
%load_ext dotenv
%dotenv 



The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [49]:
import dask.dataframe as dd
import pandas as pd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [50]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
print(f'Found {len(parquet_files)} parquet files for reading back into Dask.')

Found 2865 parquet files for reading back into Dask.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [51]:
stock_prices = dd.read_parquet(parquet_files).set_index("ticker")
stock_prices

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=90,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ALDX,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


In [ ]:
# Write your code below.

# read parket files to dask
stock_prices = dd.read_parquet(parquet_files).set_index("ticker")

# group data by ticker and calculate close lag, adj close lag, returns, hi_lo_range
dd_feat = (
    stock_prices
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending=True)
                        .assign(Close_lag_1 = x['Close'].shift(1), 
                                Adj_Close_lag_1 = x['Adj Close'].shift(1), 
                                returns = (x['Close']/(x['Close'].shift(1))) - 1,
                                hi_lo_range = x['High'] - x['Low'],
                                moving_average_returns_dd = ((x['Close']/(x['Close'].shift(1))) - 1).rolling(10).mean()), # added moving_average_returns_dd to demonstrate that moving average can be calculated in dask
            meta = pd.DataFrame(data= {'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_Close_lag_1': 'f8',
                    'returns': 'f8',
                    'hi_lo_range': 'f8',
                    'moving_average_returns_dd': 'f8'},
                    index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
        )
)
dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_average_returns_dd
npartitions=90,,,,,,,,,,,,,,
ACN,object,object,object,object,object,object,object,object,object,object,object,object,object,object
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [54]:
# Write your code below.
stock_prices_pd = dd_feat.compute()

# Calculate moving average of retuns for each ticker
stock_prices_pd['moving_average_returns_pd'] = (
    stock_prices_pd
        .groupby('ticker')['returns']
        .rolling(10)
        .mean()
        .reset_index(level=0, drop=True)
)

stock_prices_pd.head(20)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_average_returns_dd,moving_average_returns_pd
ticker,,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.930000,NaN,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,113.099998,106.026108,0.003714,1.559998,NaN,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,113.519997,106.419830,-0.007223,1.389999,NaN,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,112.699997,105.651115,0.021384,2.070000,NaN,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,115.110001,107.910400,0.003041,0.850006,NaN,NaN
ACN,2001-07-26,14.95,14.99,14.50,14.50,10.900705,6335300.0,ACN.csv,2001,115.459999,108.238510,0.009007,1.820000,NaN,NaN
ACN,2001-07-27,14.51,14.59,14.50,14.51,10.908223,3524000.0,ACN.csv,2001,116.500000,109.213455,0.002833,0.870003,NaN,NaN
ACN,2001-07-30,14.50,14.78,14.50,14.70,11.051059,3654300.0,ACN.csv,2001,116.830002,109.522820,-0.000514,1.339996,NaN,NaN
ACN,2001-07-31,14.71,15.01,14.60,14.96,11.246520,1429000.0,ACN.csv,2001,116.769997,109.466560,-0.014387,2.389999,NaN,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

As seen above, the moving average can be calculated in Dask and returns the same values, as we already sorted the data by ticker. Dask will be faster for larger amounts of data, for example the above took me 0.3s in dask, but 13.0s in pandas. If we had not taken 60 samples from the original dataset, it may not be feasible to calculate the moving average in pandas for the entire dataset. 

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.